# ResNet Model

In [ ]:
"""
ResNets50 consists of 50 layers of identity_block and Convolutional_block with skip connections
"""

# ResNet Model

In [7]:
import os 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Model
from keras.layers import Input, Conv2D, Flatten, Dropout, Dense,MaxPooling2D, AveragePooling2D,ZeroPadding2D,BatchNormalization, Activation,Add

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# Identity_block

def identity_block(X, f, filters):
 
    F1, F2, F3 = filters
    X_shortcut = X
    
    X = Conv2D(filters = F1, kernel_size = 1, strides = (1,1), padding = 'valid', kernel_initializer = keras.initializers.RandomNormal(seed=0))(X)
    X = BatchNormalization(axis = 3)(X)
    X = Activation('relu')(X)
    
    X = Conv2D(filters = F2, kernel_size = (f,f), strides = (1,1), padding = 'same', kernel_initializer = keras.initializers.RandomNormal(seed=0))(X)
    X = BatchNormalization(axis = 3)(X)
    X = Activation('relu')(X)

    X = Conv2D(filters = F3, kernel_size = 1, strides = (1,1), padding = 'valid', kernel_initializer = keras.initializers.RandomNormal(seed=0))(X)
    X = BatchNormalization(axis = 3)(X)

    
    X = Add()([X, X_shortcut])
    X = Activation('relu')(X)


    return X

In [ ]:
#Convolutional_block

def convolutional_block(X, f, filters, s = 2):

    F1, F2, F3 = filters
    X_shortcut = X

    X = Conv2D(filters = F1, kernel_size = 1, strides = (s, s), padding='valid', kernel_initializer = keras.initializers.RandomNormal(seed=0))(X)
    X = BatchNormalization(axis = 3)(X)
    X = Activation('relu')(X)

    X = Conv2D(filters = F2, kernel_size = (f,f), strides = (1, 1), padding='same', kernel_initializer = keras.initializers.RandomNormal(seed=0))(X)
    X = BatchNormalization(axis = 3)(X)
    X = Activation('relu')(X)


    X = Conv2D(filters = F3, kernel_size = 1, strides = (1, 1), padding='valid', kernel_initializer = keras.initializers.RandomNormal(seed=0))(X)
    X = BatchNormalization(axis = 3)(X)  

    X_shortcut = Conv2D(filters = F3, kernel_size = 1, strides = (s, s), padding='valid', kernel_initializer = keras.initializers.RandomNormal(seed=0))(X_shortcut)
    X_shortcut = BatchNormalization(axis = 3)(X_shortcut)
    
    X = Add()([X, X_shortcut])
    X = Activation('relu')(X)
    
    return X
    

In [ ]:
#ResNet50

def ResNet50(input_shape = (64, 64, 3), classes = 6, training=False):

    X_input = Input(input_shape)
    X = ZeroPadding2D((3, 3))(X_input) 
    
    X = Conv2D(64, (7, 7), strides = (2, 2), kernel_initializer = tf.keras.initializers.GlorotUniform(seed=0))(X)
    X = BatchNormalization(axis = 3)(X)
    X = Activation('relu')(X)
    X = MaxPooling2D((3, 3), strides=(2, 2))(X)

    X = convolutional_block(X, f = 3, filters = [64, 64, 256], s = 1)
    X = identity_block(X, 3, [64, 64, 256])
    X = identity_block(X, 3, [64, 64, 256])

    X = convolutional_block(X, f = 3, filters = [128,128,512], s = 2)
    X = identity_block(X, 3, [128,128,512])
    X = identity_block(X, 3, [128,128,512])
    X = identity_block(X, 3, [128,128,512])
    
    X = convolutional_block(X, f = 3, filters = [256, 256, 1024], s = 2)
    X = identity_block(X, 3, [256, 256, 1024])
    X = identity_block(X, 3, [256, 256, 1024])
    X = identity_block(X, 3, [256, 256, 1024])
    X = identity_block(X, 3, [256, 256, 1024])
    X = identity_block(X, 3, [256, 256, 1024])

    X = convolutional_block(X, f = 3, filters = [512, 512, 2048], s = 2)
    X = identity_block(X, 3, [512, 512, 2048])
    X = identity_block(X, 3, [512, 512, 2048])
    X = AveragePooling2D((2, 2))(X)
    
    X = Flatten()(X)
    X = Dense(classes, activation='softmax', kernel_initializer = tf.keras.initializers.GlorotUniform(seed=0))(X)
    
    model = Model(inputs = X_input, outputs = X)

    return model

In [ ]:
model = ResNet50(input_shape = (64, 64, 3), classes = 10)
print(model.summary())

In [ ]:
for i, layer in enumerate(model.layers):
        print(i,layer.name,layer.count_params())
    